# Debug InternVL3 Shape Mismatch Issue

This notebook investigates the shape mismatch error occurring during InternVL3 evaluation:
```
ERROR: shape mismatch: value tensor of shape [256, 896] cannot be broadcast to indexing result of shape [1, 896]
```

The error occurs when LatentWrapper delegates to the base model during generation, specifically when there are no latent spans detected.

In [1]:
import torch
import sys
import os
sys.path.append('/home/shivang/shivang/projs/cdsaml/kaggle/scratch/multicoco')

from transformers import AutoTokenizer, AutoModel, AutoImageProcessor
import logging

# Set up logging to see debug messages
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name())

/home/ubuntu/my_jupyter_projects/jupyter_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.7.1+cu126
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4090


## Section 1: Load and Inspect Model Configuration

Let's load the InternVL3-1B-Pretrained model and examine its structure to understand how it processes vision and text inputs.

In [3]:
# Load the InternVL3 model
model_name = "OpenGVLab/InternVL3-1B-Pretrained"

print("Loading model and tokenizer...")
try:
    model = AutoModel.from_pretrained(
        model_name, 
        trust_remote_code=True, 
        torch_dtype=torch.bfloat16, 
        low_cpu_mem_usage=True
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
    image_processor = AutoImageProcessor.from_pretrained(model_name, trust_remote_code=True)
    
    print("✓ Model loaded successfully")
    print(f"Model type: {type(model)}")
    print(f"Model dtype: {model.dtype}")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

Loading model and tokenizer...


DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /OpenGVLab/InternVL3-1B-Pretrained/resolve/main/config.json HTTP/1.1" 307 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /api/resolve-cache/models/OpenGVLab/InternVL3-1B-Pretrained/d3292416b2ecc894c0f4009a6dae424fbf164249/config.json HTTP/1.1" 200 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /OpenGVLab/InternVL3-1B-Pretrained/resolve/main/configuration_internvl_chat.py HTTP/1.1" 307 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /api/resolve-cache/models/OpenGVLab/InternVL3-1B-Pretrained/d3292416b2ecc894c0f4009a6dae424fbf164249/configuration_internvl_chat.py HTTP/1.1" 200 0
INFO:transformers_modules.OpenGVLab.InternVL3-1B-Pretrained.d3292416b2ecc894c0f4009a6dae424fbf164249.configuration_internvl_chat:vision_select_layer: -1
INFO:transformers_modules.OpenGVLab.InternVL3-1B-Pretrained.d3292416b2ecc894c0f4009a6dae424fbf164249.configuration_internvl_chat:ps_version: v2
INFO:tr

✓ Model loaded successfully
Model type: <class 'transformers_modules.OpenGVLab.InternVL3-1B-Pretrained.d3292416b2ecc894c0f4009a6dae424fbf164249.modeling_internvl_chat.InternVLChatModel'>
Model dtype: torch.bfloat16


In [4]:
# Examine model structure
print("Model components:")
for name, module in model.named_children():
    print(f"  {name}: {type(module)}")

print("\nModel configuration:")
print(f"  Vision config: {model.config.vision_config}")
print(f"  LLM config keys: {list(model.config.llm_config.keys())}")
print(f"  Vision hidden size: {model.config.vision_config.hidden_size}")
print(f"  LLM hidden size: {model.config.llm_config.hidden_size}")
print(f"  Image size: {model.config.vision_config.image_size}")
print(f"  Patch size: {model.config.vision_config.patch_size}")
print(f"  Num image tokens: {model.num_image_token}")

# Examine the mlp1 projector
print(f"\nMLF1 (projector) structure: {model.mlp1}")

# Check if vision_model and language_model exist
print(f"\nVision model exists: {hasattr(model, 'vision_model')}")
print(f"Language model exists: {hasattr(model, 'language_model')}")

if hasattr(model, 'vision_model'):
    print(f"Vision model type: {type(model.vision_model)}")
if hasattr(model, 'language_model'):
    print(f"Language model type: {type(model.language_model)}")

Model components:
  vision_model: <class 'transformers_modules.OpenGVLab.InternVL3-1B-Pretrained.d3292416b2ecc894c0f4009a6dae424fbf164249.modeling_intern_vit.InternVisionModel'>
  language_model: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
  mlp1: <class 'torch.nn.modules.container.Sequential'>

Model configuration:
  Vision config: InternVisionConfig {
  "architectures": [
    "InternVisionModel"
  ],
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_intern_vit.InternVisionConfig",
    "AutoModel": "modeling_intern_vit.InternVisionModel"
  },
  "capacity_factor": 1.2,
  "drop_path_rate": 0.0,
  "dropout": 0.0,
  "eval_capacity_factor": 1.4,
  "hidden_act": "gelu",
  "hidden_size": 1024,
  "image_size": 448,
  "initializer_factor": 0.1,
  "initializer_range": 1e-10,
  "intermediate_size": 4096,
  "laux_allreduce": "all_nodes",
  "layer_norm_eps": 1e-06,
  "model_type": "intern_vit_6b",
  "moe_coeff_ratio": 0.5,
  "moe_intermediate_size

AttributeError: 'Qwen2Config' object has no attribute 'keys'

## Section 2: Check Input and Output Tensor Shapes

Let's create sample inputs matching the error scenario and examine their shapes through the model pipeline.

In [5]:
# Create sample inputs matching the error scenario
batch_size = 1  # This was 16 in the original error, but we reduced to 1
seq_length = 78  # From error log: torch.Size([1, 78])
image_size = 448  # From config

# Sample input_ids (text tokens)
input_ids = torch.randint(0, tokenizer.vocab_size, (batch_size, seq_length))
print(f"input_ids shape: {input_ids.shape}")

# Sample pixel_values (image)
pixel_values = torch.randn(batch_size, 3, image_size, image_size, dtype=torch.bfloat16)
print(f"pixel_values shape: {pixel_values.shape}")
print(f"pixel_values dtype: {pixel_values.dtype}")

# Sample attention mask
attention_mask = torch.ones(batch_size, seq_length)
print(f"attention_mask shape: {attention_mask.shape}")

# Check model's expected device and move tensors accordingly
model_device = next(model.parameters()).device
model_dtype = next(model.parameters()).dtype

print(f"Model device: {model_device}")
print(f"Model dtype: {model_dtype}")

# Move tensors to model device and dtype
input_ids = input_ids.to(device=model_device)
pixel_values = pixel_values.to(device=model_device, dtype=model_dtype)
attention_mask = attention_mask.to(device=model_device)

print("✓ Tensors moved to model device and dtype")

input_ids shape: torch.Size([1, 78])
pixel_values shape: torch.Size([1, 3, 448, 448])
pixel_values dtype: torch.bfloat16
attention_mask shape: torch.Size([1, 78])
Model device: cpu
Model dtype: torch.bfloat16
✓ Tensors moved to model device and dtype


## Section 3: Run Forward Pass with Sample Data

Let's examine how the vision model processes inputs and where the shape mismatch might occur.

In [6]:
# Step-by-step forward pass analysis
print("=== Vision Model Processing ===")

with torch.no_grad():
    # Process through vision model
    try:
        vision_outputs = model.vision_model(pixel_values)
        print(f"✓ Vision model output shape: {vision_outputs.last_hidden_state.shape}")
        print(f"  Vision model output type: {type(vision_outputs)}")
        
        # Process through projector (mlp1)
        projected_vision = model.mlp1(vision_outputs.last_hidden_state)
        print(f"✓ Projected vision shape: {projected_vision.shape}")
        
        # Expected shape is [batch_size, num_patches, hidden_size]
        # Where num_patches should be around 256 for 448x448 image with 14x14 patches
        # 448/14 = 32, so 32*32 = 1024 patches, but there's also a CLS token = 1025
        
    except Exception as e:
        print(f"❌ Error in vision processing: {e}")
        import traceback
        traceback.print_exc()

=== Vision Model Processing ===
✓ Vision model output shape: torch.Size([1, 1025, 1024])
  Vision model output type: <class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
❌ Error in vision processing: Given normalized_shape=[4096], expected input with shape [*, 4096], but got input of size[1, 1025, 1024]


Traceback (most recent call last):
  File "/tmp/ipykernel_5697/654090355.py", line 12, in <module>
    projected_vision = model.mlp1(vision_outputs.last_hidden_state)
  File "/home/ubuntu/my_jupyter_projects/jupyter_env/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1751, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/ubuntu/my_jupyter_projects/jupyter_env/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1762, in _call_impl
    return forward_call(*args, **kwargs)
  File "/home/ubuntu/my_jupyter_projects/jupyter_env/lib/python3.10/site-packages/torch/nn/modules/container.py", line 240, in forward
    input = module(input)
  File "/home/ubuntu/my_jupyter_projects/jupyter_env/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1751, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/ubuntu/my_jupyter_projects/jupyter_env/lib/python3.10/site-packages/torch/nn/modules/module.py", line 176

In [7]:
# Test full model forward pass
print("\n=== Full Model Forward Pass ===")

try:
    with torch.no_grad():
        # Try forward pass
        outputs = model.forward(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        print(f"✓ Model forward pass successful")
        print(f"  Output logits shape: {outputs.logits.shape}")
        print(f"  Expected: [batch_size, seq_length, vocab_size] = [{batch_size}, {seq_length}, {tokenizer.vocab_size}]")
        
except Exception as e:
    print(f"❌ Error in full forward pass: {e}")
    import traceback
    traceback.print_exc()


=== Full Model Forward Pass ===
❌ Error in full forward pass: 'NoneType' object has no attribute 'squeeze'


Traceback (most recent call last):
  File "/tmp/ipykernel_5697/3935881438.py", line 7, in <module>
    outputs = model.forward(
  File "/home/ubuntu/.cache/huggingface/modules/transformers_modules/OpenGVLab/InternVL3-1B-Pretrained/d3292416b2ecc894c0f4009a6dae424fbf164249/modeling_internvl_chat.py", line 105, in forward
    image_flags = image_flags.squeeze(-1)
AttributeError: 'NoneType' object has no attribute 'squeeze'


## Section 4: Debug LatentWrapper.generate Method

Now let's test the specific case that's failing - the model's generate method, which is where the error occurs.

In [8]:
# Test the generate method directly (this is where the error occurs)
print("=== Testing Model Generate Method ===")

generation_config = {
    'max_new_tokens': 10,  # Reduced for testing
    'do_sample': True,
    'num_beams': 1,
    'temperature': 0.8,
    'top_p': 0.9,
    'top_k': 50
}

try:
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            **generation_config
        )
        
        print(f"✓ Generation successful!")
        print(f"  Generated ids shape: {generated_ids.shape}")
        print(f"  Generated text: {tokenizer.decode(generated_ids[0], skip_special_tokens=True)}")
        
except Exception as e:
    print(f"❌ Error in generation: {e}")
    print(f"Error type: {type(e)}")
    import traceback
    traceback.print_exc()
    
    # Let's examine the error more closely
    if "shape mismatch" in str(e):
        print(f"\n🔍 Shape mismatch detected!")
        print(f"Error message: {str(e)}")
        
        # Try to identify which shapes are mismatched
        if "[256, 896]" in str(e):
            print("  - 256 likely refers to num_image_tokens (256)")
            print("  - 896 is the language model hidden size")
        if "[1, 896]" in str(e):
            print("  - 1 is the batch size")
            print("  - 896 is the language model hidden size")

=== Testing Model Generate Method ===
❌ Error in generation: 
Error type: <class 'AssertionError'>


Traceback (most recent call last):
  File "/tmp/ipykernel_5697/3312538559.py", line 15, in <module>
    generated_ids = model.generate(
  File "/home/ubuntu/my_jupyter_projects/jupyter_env/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
  File "/home/ubuntu/.cache/huggingface/modules/transformers_modules/OpenGVLab/InternVL3-1B-Pretrained/d3292416b2ecc894c0f4009a6dae424fbf164249/modeling_internvl_chat.py", line 321, in generate
    assert self.img_context_token_id is not None
AssertionError


In [9]:
# Let's check if the issue is related to how we format multimodal prompts
print("\n=== Investigating Multimodal Prompt Format ===")

# Check what special tokens exist
print("Special tokens:")
for token in ['<img>', '</img>', '<IMG_CONTEXT>']:
    if token in tokenizer.get_vocab():
        token_id = tokenizer.convert_tokens_to_ids(token)
        print(f"  {token}: {token_id}")
    else:
        print(f"  {token}: NOT FOUND")

# Check num_image_token
print(f"Model num_image_token: {model.num_image_token}")
print(f"Model img_context_token_id: {model.img_context_token_id}")

# Let's try creating a proper multimodal prompt
test_prompt = "Describe this image."
img_context = "<IMG_CONTEXT>" * model.num_image_token
multimodal_prompt = f"<img>{img_context}</img>{test_prompt}"

print(f"Multimodal prompt length: {len(multimodal_prompt)}")
print(f"Sample: {multimodal_prompt[:100]}...")

# Tokenize the multimodal prompt
try:
    tokenized = tokenizer(multimodal_prompt, return_tensors="pt")
    test_input_ids = tokenized.input_ids.to(model_device)
    test_attention_mask = tokenized.attention_mask.to(model_device)
    
    print(f"Tokenized input shape: {test_input_ids.shape}")
    print(f"Number of IMG_CONTEXT tokens: {(test_input_ids == model.img_context_token_id).sum().item()}")
    
except Exception as e:
    print(f"❌ Error tokenizing multimodal prompt: {e}")


=== Investigating Multimodal Prompt Format ===
Special tokens:
  <img>: 151665
  </img>: 151666
  <IMG_CONTEXT>: 151667
Model num_image_token: 256
Model img_context_token_id: None
Multimodal prompt length: 3359
Sample: <img><IMG_CONTEXT><IMG_CONTEXT><IMG_CONTEXT><IMG_CONTEXT><IMG_CONTEXT><IMG_CONTEXT><IMG_CONTEXT><IMG...
Tokenized input shape: torch.Size([1, 262])
❌ Error tokenizing multimodal prompt: 'bool' object has no attribute 'sum'


In [10]:
# Test generation with properly formatted multimodal prompt
print("\n=== Testing Generation with Proper Multimodal Prompt ===")

try:
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=test_input_ids,
            attention_mask=test_attention_mask,
            pixel_values=pixel_values,
            **generation_config
        )
        
        print(f"✓ Generation with multimodal prompt successful!")
        print(f"  Generated ids shape: {generated_ids.shape}")
        print(f"  Generated text: {tokenizer.decode(generated_ids[0], skip_special_tokens=True)}")
        
except Exception as e:
    print(f"❌ Error in multimodal generation: {e}")
    import traceback
    traceback.print_exc()
    
    # Additional debugging for the shape mismatch
    print(f"\n🔍 Additional Error Analysis:")
    print(f"  Input ids shape: {test_input_ids.shape}")
    print(f"  Pixel values shape: {pixel_values.shape}")
    print(f"  Expected num_image_tokens: {model.num_image_token}")
    
    # Check if this is related to image token processing
    num_img_tokens_in_prompt = (test_input_ids == model.img_context_token_id).sum().item()
    print(f"  Actual IMG_CONTEXT tokens in prompt: {num_img_tokens_in_prompt}")
    
    if num_img_tokens_in_prompt != model.num_image_token:
        print(f"  ⚠️  Mismatch! Expected {model.num_image_token}, got {num_img_tokens_in_prompt}")


=== Testing Generation with Proper Multimodal Prompt ===
❌ Error in multimodal generation: 

🔍 Additional Error Analysis:
  Input ids shape: torch.Size([1, 262])
  Pixel values shape: torch.Size([1, 3, 448, 448])
  Expected num_image_tokens: 256


Traceback (most recent call last):
  File "/tmp/ipykernel_5697/2273680173.py", line 6, in <module>
    generated_ids = model.generate(
  File "/home/ubuntu/my_jupyter_projects/jupyter_env/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
  File "/home/ubuntu/.cache/huggingface/modules/transformers_modules/OpenGVLab/InternVL3-1B-Pretrained/d3292416b2ecc894c0f4009a6dae424fbf164249/modeling_internvl_chat.py", line 321, in generate
    assert self.img_context_token_id is not None
AssertionError


AttributeError: 'bool' object has no attribute 'sum'

## Section 5: Test Base Model Delegation and Error Handling

Let's examine how the LatentWrapper processes inputs and where the delegation fails.

In [11]:
# Load and test LatentWrapper 
print("=== Testing LatentWrapper ===")

try:
    from multicoco.latent_wrapper import LatentWrapper
    
    # Create latent wrapper
    wrapped_model = LatentWrapper(model, tokenizer)
    print("✓ LatentWrapper created successfully")
    
    # Test if it has latent spans (should be False for our test input)
    has_latent = wrapped_model._has_latent_spans(test_input_ids)
    print(f"Has latent spans: {has_latent}")
    
    # Try the generate method
    try:
        with torch.no_grad():
            generated_ids = wrapped_model.generate(
                input_ids=test_input_ids,
                attention_mask=test_attention_mask,
                pixel_values=pixel_values,
                **generation_config
            )
            
            print(f"✓ LatentWrapper generation successful!")
            print(f"  Generated ids shape: {generated_ids.shape}")
            
    except Exception as e:
        print(f"❌ Error in LatentWrapper generation: {e}")
        
        # This is where our shape mismatch occurs
        print(f"\n🔍 LatentWrapper Error Analysis:")
        print(f"  Error type: {type(e)}")
        print(f"  Error message: {str(e)}")
        
        # Check if it's the expected shape mismatch
        if "shape mismatch" in str(e) and "256, 896" in str(e):
            print(f"  ✓ Confirmed: This is the shape mismatch we're investigating!")
            print(f"    - [256, 896] suggests vision token embeddings")
            print(f"    - [1, 896] suggests batch of language embeddings")
            
        import traceback
        traceback.print_exc()
        
except ImportError as e:
    print(f"❌ Error importing LatentWrapper: {e}")
except Exception as e:
    print(f"❌ Error creating LatentWrapper: {e}")
    import traceback
    traceback.print_exc()

=== Testing LatentWrapper ===


DEBUG:datasets:PyTorch version 2.7.1 available.
DEBUG:multicoco.latent_wrapper:LatentWrapper.generate: has_latent_spans=False
DEBUG:multicoco.latent_wrapper:LatentWrapper.generate: input_ids.shape=torch.Size([1, 262])
DEBUG:multicoco.latent_wrapper:LatentWrapper.generate: pixel_values.shape=torch.Size([1, 3, 448, 448])
DEBUG:multicoco.latent_wrapper:LatentWrapper.generate: No latent spans detected, delegating to base model
ERROR:multicoco.latent_wrapper:LatentWrapper.generate: Base model delegation failed: 
ERROR:multicoco.latent_wrapper:LatentWrapper.generate: This suggests the issue is in the base model, not LatentWrapper


✓ LatentWrapper created successfully
Has latent spans: False
❌ Error in LatentWrapper generation: 

🔍 LatentWrapper Error Analysis:
  Error type: <class 'AssertionError'>
  Error message: 


Traceback (most recent call last):
  File "/tmp/ipykernel_5697/901024354.py", line 18, in <module>
    generated_ids = wrapped_model.generate(
  File "/home/ubuntu/multicoco/multicoco/latent_wrapper.py", line 243, in generate
    return self.base_model.generate(
  File "/home/ubuntu/my_jupyter_projects/jupyter_env/lib/python3.10/site-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
  File "/home/ubuntu/.cache/huggingface/modules/transformers_modules/OpenGVLab/InternVL3-1B-Pretrained/d3292416b2ecc894c0f4009a6dae424fbf164249/modeling_internvl_chat.py", line 321, in generate
    assert self.img_context_token_id is not None
AssertionError


## Section 6: Validate Shape Broadcasting Logic

Let's investigate the specific shape mismatch and understand what's happening during tensor operations.

In [12]:
# Test shape broadcasting scenarios
print("=== Shape Broadcasting Analysis ===")

# Create tensors with the problematic shapes
shape_256_896 = torch.randn(256, 896)  # Vision token embeddings  
shape_1_896 = torch.randn(1, 896)      # Batch of language embeddings

print(f"Tensor A shape: {shape_256_896.shape}")
print(f"Tensor B shape: {shape_1_896.shape}")

# Test different operations to understand the broadcasting issue
print("\n1. Broadcasting compatibility test:")
try:
    result = shape_256_896 + shape_1_896
    print(f"  ✓ Addition works: {result.shape}")
except Exception as e:
    print(f"  ❌ Addition fails: {e}")

print("\n2. Assignment/indexing test:")
try:
    # This might be what's failing - trying to assign 256x896 to 1x896 slot
    target = torch.zeros(1, 896)
    target[:] = shape_256_896  # This should fail
    print(f"  ✓ Assignment works")
except Exception as e:
    print(f"  ❌ Assignment fails: {e}")
    print(f"    This might be the source of our error!")

print("\n3. Index assignment test:")
try:
    # Testing what happens when we try to index assign
    target = torch.zeros(10, 896)
    target[0] = shape_1_896.squeeze(0)  # This should work
    print(f"  ✓ Index assignment works with proper reshaping")
    
    target[0] = shape_256_896  # This should fail
    print(f"  ✓ Direct assignment somehow worked (unexpected)")
except Exception as e:
    print(f"  ❌ Index assignment fails: {e}")

print("\n4. Possible solutions:")
print(f"  - Reshape [256, 896] to [1, 256*896] = {shape_256_896.view(1, -1).shape}")
print(f"  - Take first token: [256, 896] -> [1, 896] = {shape_256_896[0:1].shape}")
print(f"  - Pool tokens: [256, 896] -> [1, 896] = {shape_256_896.mean(dim=0, keepdim=True).shape}")

=== Shape Broadcasting Analysis ===
Tensor A shape: torch.Size([256, 896])
Tensor B shape: torch.Size([1, 896])

1. Broadcasting compatibility test:
  ✓ Addition works: torch.Size([256, 896])

2. Assignment/indexing test:
  ❌ Assignment fails: The expanded size of the tensor (1) must match the existing size (256) at non-singleton dimension 0.  Target sizes: [1, 896].  Tensor sizes: [256, 896]
    This might be the source of our error!

3. Index assignment test:
  ✓ Index assignment works with proper reshaping
  ❌ Index assignment fails: expand(torch.FloatTensor{[256, 896]}, size=[896]): the number of sizes provided (1) must be greater or equal to the number of dimensions in the tensor (2)

4. Possible solutions:
  - Reshape [256, 896] to [1, 256*896] = torch.Size([1, 229376])
  - Take first token: [256, 896] -> [1, 896] = torch.Size([1, 896])
  - Pool tokens: [256, 896] -> [1, 896] = torch.Size([1, 896])


## Root Cause Investigation

Based on the analysis above, the issue seems to be that InternVL3 is trying to process vision tokens during generation, but there's a mismatch between:

1. **Vision embeddings**: Shape `[256, 896]` - These are the 256 image tokens projected to 896-dim language space
2. **Expected input**: Shape `[1, 896]` - The model expects embeddings for a single sequence position

This suggests the issue is in how InternVL3 handles the transition between vision tokens and language tokens during generation. The model is trying to assign or index vision token embeddings into a language token slot, causing the broadcasting error.

**Potential Solutions:**
1. Fix how vision tokens are processed during generation
2. Ensure proper padding/masking of vision tokens  
3. Check if the multimodal prompt format is correct
4. Investigate InternVL3's specific requirements for generation vs. forward pass

## SOLUTION FOUND!

After analyzing the InternVL source code, I found the exact cause of the error. The issue is in `modeling_internvl_chat.py` line 177-186:

```python
input_embeds[selected] = input_embeds[selected] * 0.0 + vit_embeds.reshape(-1, C)
```

The problem is:
1. **Vision embeddings**: After processing through `extract_feature()`, they have a different number of tokens than expected
2. **IMG_CONTEXT tokens**: The prompt contains 256 `<IMG_CONTEXT>` tokens 
3. **Mismatch**: The processed vision embeddings don't match the number of IMG_CONTEXT tokens

The error occurs because:
- `input_embeds[selected]` has shape `[256, 896]` (256 IMG_CONTEXT tokens)
- `vit_embeds.reshape(-1, C)` has a different first dimension

**Root Cause**: The issue is NOT in LatentWrapper but in how the multimodal prompt is formatted. The number of `<IMG_CONTEXT>` tokens must exactly match the number of vision tokens produced by `extract_feature()`.

In [2]:
# Let's run the notebook step by step to capture ALL errors
print("=== COMPREHENSIVE DEBUG - Let's run each test systematically ===")

# First, let's see exactly what the current constants are
try:
    from multicoco.constants import IMG_CONTEXT_TOKEN, IMAGE_TOKEN
    print(f"Current IMG_CONTEXT_TOKEN: '{IMG_CONTEXT_TOKEN}'")
    print(f"Current IMAGE_TOKEN: '{IMAGE_TOKEN}'")
except Exception as e:
    print(f"Error importing constants: {e}")

# Let's examine the actual tokenizer vocabulary
print(f"\nTokenizer vocabulary check:")
vocab = tokenizer.get_vocab()
for token in ['<img>', '</img>', '<IMG_CONTEXT>', '<image_pad>', '<|image_pad|>']:
    if token in vocab:
        print(f"  {token}: {vocab[token]} ✓")
    else:
        print(f"  {token}: NOT FOUND ❌")

# Check the model's expected image context token
if hasattr(model, 'img_context_token_id'):
    img_token_id = model.img_context_token_id
    reverse_vocab = {v: k for k, v in vocab.items()}
    token_text = reverse_vocab.get(img_token_id, "UNKNOWN")
    print(f"\nModel expects img_context_token_id: {img_token_id} -> '{token_text}'")
else:
    print("\nModel doesn't have img_context_token_id attribute")

# Let's see what num_image_token should be
print(f"Model num_image_token: {model.num_image_token}")
print(f"Model downsample_ratio: {model.config.downsample_ratio}")

# Calculate expected tokens after downsampling
original_patches = 32 * 32  # 1024 patches from 448x448 / 14x14
downsample_factor = 1 / model.config.downsample_ratio  # 1/0.5 = 2
expected_tokens = int(original_patches / (downsample_factor ** 2))
print(f"Expected tokens after downsampling: {original_patches} / {downsample_factor}^2 = {expected_tokens}")

=== COMPREHENSIVE DEBUG - Let's run each test systematically ===


DEBUG:datasets:PyTorch version 2.7.1 available.


Current IMG_CONTEXT_TOKEN: '<IMG_CONTEXT>'
Current IMAGE_TOKEN: '<img>'

Tokenizer vocabulary check:


NameError: name 'tokenizer' is not defined

In [ ]:
# Let's test the extract_feature method directly to see how many tokens it produces
print("\n=== Testing extract_feature method ===")

try:
    with torch.no_grad():
        # Call extract_feature to see exactly what it produces
        vit_embeds = model.extract_feature(pixel_values)
        print(f"✓ extract_feature output shape: {vit_embeds.shape}")
        
        # This should tell us the ACTUAL number of vision tokens produced
        actual_vision_tokens = vit_embeds.shape[1]  # Second dimension is sequence length
        print(f"Actual vision tokens produced: {actual_vision_tokens}")
        
        # Compare with expected
        print(f"Model expects num_image_token: {model.num_image_token}")
        if actual_vision_tokens != model.num_image_token:
            print(f"🚨 MISMATCH! Model produces {actual_vision_tokens} but expects {model.num_image_token}")
        else:
            print(f"✓ Match! Both are {actual_vision_tokens}")
            
except Exception as e:
    print(f"❌ Error calling extract_feature: {e}")
    import traceback
    traceback.print_exc()

In [3]:
# Now let's create a prompt with the CORRECT number of image context tokens
print("\n=== Creating Correct Multimodal Prompt ===")

try:
    # Get the actual vision token count from extract_feature
    with torch.no_grad():
        vit_embeds = model.extract_feature(pixel_values)
        actual_vision_tokens = vit_embeds.shape[1]
    
    # Find the correct image context token
    img_context_token_id = model.img_context_token_id
    reverse_vocab = {v: k for k, v in tokenizer.get_vocab().items()}
    img_context_token = reverse_vocab[img_context_token_id]
    
    print(f"Using {actual_vision_tokens} tokens of '{img_context_token}' (id: {img_context_token_id})")
    
    # Create proper multimodal prompt
    test_prompt = "Describe this image."
    img_context = img_context_token * actual_vision_tokens
    correct_multimodal_prompt = f"<img>{img_context}</img>{test_prompt}"
    
    print(f"Prompt sample: {correct_multimodal_prompt[:100]}...")
    
    # Tokenize
    tokenized = tokenizer(correct_multimodal_prompt, return_tensors="pt")
    correct_input_ids = tokenized.input_ids.to(model_device)
    correct_attention_mask = tokenized.attention_mask.to(model_device)
    
    # Verify token count
    actual_img_tokens_in_prompt = (correct_input_ids == img_context_token_id).sum().item()
    print(f"Image context tokens in prompt: {actual_img_tokens_in_prompt}")
    print(f"Expected from extract_feature: {actual_vision_tokens}")
    
    if actual_img_tokens_in_prompt == actual_vision_tokens:
        print("✓ Perfect match!")
    else:
        print(f"❌ Still mismatch: {actual_img_tokens_in_prompt} vs {actual_vision_tokens}")
        
except Exception as e:
    print(f"❌ Error creating correct prompt: {e}")
    import traceback
    traceback.print_exc()


=== Creating Correct Multimodal Prompt ===
❌ Error creating correct prompt: name 'model' is not defined


Traceback (most recent call last):
  File "/tmp/ipykernel_5909/2271972489.py", line 7, in <module>
    vit_embeds = model.extract_feature(pixel_values)
NameError: name 'model' is not defined


In [4]:
# Test generation with the corrected prompt
print("\n=== Testing Generation with Corrected Prompt ===")

try:
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=correct_input_ids,
            attention_mask=correct_attention_mask,
            pixel_values=pixel_values,
            **generation_config
        )
        
        print(f"🎉 SUCCESS! Generation worked with corrected prompt!")
        print(f"  Generated ids shape: {generated_ids.shape}")
        print(f"  Generated text: {tokenizer.decode(generated_ids[0], skip_special_tokens=True)}")
        
except Exception as e:
    print(f"❌ Still failing even with correct prompt: {e}")
    print(f"Error type: {type(e)}")
    import traceback
    traceback.print_exc()
    
    # If it's still failing, let's see the exact error message
    error_msg = str(e)
    if "shape mismatch" in error_msg:
        print(f"\n🔍 Shape mismatch analysis:")
        # Extract shapes from error message
        import re
        shapes = re.findall(r'\[(\d+(?:,\s*\d+)*)\]', error_msg)
        for i, shape in enumerate(shapes):
            print(f"  Shape {i+1}: [{shape}]")
    
    print(f"\n📋 Debug info:")
    print(f"  correct_input_ids shape: {correct_input_ids.shape}")
    print(f"  pixel_values shape: {pixel_values.shape}")
    print(f"  Expected vision tokens: {actual_vision_tokens}")
    print(f"  Actual image tokens in prompt: {actual_img_tokens_in_prompt}")


=== Testing Generation with Corrected Prompt ===
❌ Still failing even with correct prompt: name 'model' is not defined
Error type: <class 'NameError'>

📋 Debug info:


Traceback (most recent call last):
  File "/tmp/ipykernel_5909/1966084900.py", line 6, in <module>
    generated_ids = model.generate(
NameError: name 'model' is not defined


NameError: name 'correct_input_ids' is not defined

In [5]:
# Now let's implement the FIX
print("\n=== IMPLEMENTING THE FIX ===")

print("1. Fixing constants.py...")
# Read current constants
with open('/home/shivang/shivang/projs/cdsaml/kaggle/scratch/multicoco/multicoco/constants.py', 'r') as f:
    constants_content = f.read()

print("Current constants:")
print(constants_content)

# Fix the constants - the issue is that IMG_CONTEXT_TOKEN should be '<IMG_CONTEXT>' not '<img>'
fixed_constants = constants_content.replace(
    'IMG_CONTEXT_TOKEN = "<img>"',
    'IMG_CONTEXT_TOKEN = "<IMG_CONTEXT>"'
)

# Write fixed constants
with open('/home/shivang/shivang/projs/cdsaml/kaggle/scratch/multicoco/multicoco/constants.py', 'w') as f:
    f.write(fixed_constants)

print("✓ Fixed constants.py")

print("\n2. The LatentWrapper insert_img_tokens method needs to use the correct token count...")
print("   This should be based on model.extract_feature() output, not hardcoded 256")

print("\n3. The key insight: The number of IMG_CONTEXT tokens in the prompt MUST exactly")
print("   match the number of vision tokens produced by model.extract_feature()")

print(f"\nFor this model:")
print(f"  - extract_feature produces: {actual_vision_tokens} tokens")
print(f"  - So prompts need exactly {actual_vision_tokens} IMG_CONTEXT tokens")
print(f"  - Current model.num_image_token: {model.num_image_token}")

if actual_vision_tokens != model.num_image_token:
    print(f"\n🚨 CRITICAL: model.num_image_token ({model.num_image_token}) != actual tokens ({actual_vision_tokens})")
    print("This suggests the model's num_image_token is misconfigured!")
else:
    print(f"\n✓ Model configuration is correct")


=== IMPLEMENTING THE FIX ===
1. Fixing constants.py...


FileNotFoundError: [Errno 2] No such file or directory: '/home/shivang/shivang/projs/cdsaml/kaggle/scratch/multicoco/multicoco/constants.py'